# 小配置头上的固定分钟数值增量

可用性对照与原12核心分钟字段整组加入，三seed、300轮、固定完整成本账户。保留强裸小模型参考，不筛单项或修改窗口。

In [1]:
from pathlib import Path
import os,json
import qlib
from src.workflow.small_head_minutes import run
root=Path.cwd()
while not (root/'SPEC.md').exists():root=root.parent
os.chdir(root);os.environ['MLFLOW_ALLOW_FILE_STORE']='true';card=root/'research/small_head_minute_increment_20261014';plan=json.loads((card/'plan.json').read_text());qlib.init(provider_uri=str(root/'data/qlib/full_path_tiered_allocation_20261013'),region='us',kernels=1)
summary=run(root=root,card=card,plan=plan)
print({k:summary[k] for k in ['decision','new_fits','new_native_cases','gates']})


In [1]:
from pathlib import Path
import json,hashlib,numpy as np,pandas as pd,qlib
from qlib.workflow import R
from src.data.minute_intake import atomic_json
from src.data.nas_market_data import archive_nas_recorder
root=Path.cwd();card=root/'research/small_head_minute_increment_20261014';s=json.loads((card/'summary.json').read_text());plan=json.loads((card/'plan.json').read_text());qlib.init(provider_uri=s['provider_uri'],region='us',kernels=1);rows=[]
with R.uri_context(s['recorder_uri']):
 rec=R.get_recorder(recorder_id=s['recorder_id'],experiment_name=card.name)
 for seed in plan['seeds']:
  model=rec.load_object('models/candidate_'+str(seed));w=model.network.allocation_head.weight.detach().cpu().numpy();offset=len(model.base_features)+len(plan['availability_features'])
  for j,name in enumerate(plan['core_features']):
   scale=model.extra_scaler[name];v=w[:,offset+j];centered=v-v.mean();rows.append({'seed':seed,'feature':name,'active':bool(scale['active']),'known_training_rows':scale['observations'],'relative_logit_coefficient_abs_sum':float(np.abs(centered).sum())})
 usage=pd.DataFrame(rows);s.update(numeric_feature_count=len(plan['core_features']),numeric_channels_active_all_seeds=bool(usage.active.all()),numeric_channels_nonzero_relative_coefficients_all_seeds=bool(usage.relative_logit_coefficient_abs_sum.gt(0).all()),scope=s['scope']+' Numeric channels are verified active with learned relative-logit coefficients; coefficients are not evidence of profitable causality.')
 with R.start(experiment_name=card.name,recorder_id=s['recorder_id'],resume=True):
  rec=R.get_recorder();rec.save_objects(artifact_path='analysis',feature_usage=usage,summary=s);artifacts=Path(s['recorder_local_dir'])/'artifacts';rec.log_metrics(registered_artifact_count=sum(p.is_file() for p in artifacts.rglob('*')))
atomic_json({'recorder_id':rec.id,'entries':[{'path':str(p.relative_to(card)),'sha256':hashlib.sha256(p.read_bytes()).hexdigest()} for p in sorted(artifacts.rglob('*')) if p.is_file()]},card/'verification_manifest.json');s['nas_evidence']=archive_nas_recorder(card/'nas_binding.json',workspace_root=root,card=card,recorder_id=rec.id,experiment_id=s['experiment_id']);atomic_json(s,card/'summary.json');usage.to_csv(card/'feature_usage.csv',index=False)


# 小配置头的分钟数值增量：未通过强参考与压力费用门

本轮实际完成6次训练：可用性对照和分钟数值候选各3个固定seed，每个3014主训练样本、20741辅助样本、300轮；固定最终轮、等权平均。新增6套Qlib完整原生账户，复用12套强参考。没有单项相关性筛选、窗口调整或最佳seed选择。

两组都用已冻结的小配置头。对照只加minute_core_ready/minute_peer_ready两个标记，候选再加原先定义的全部12项上一完整交易日核心分钟特征。配置主输入41→53，共同编码器仍16项输入，网络参数304→340。新增数值系数初始为0，三个seed初始logits完全一致；非新增系数、分类目标、损失、训练切分和执行控制器不变。增加特征同时增加36个参数，不能说参数量完全相同。

分钟数据确实进入训练：12项在三个seed均激活=True；每项已知训练观察1055日，均学到非零相对logit系数=True。更早缺失保持NaN，模型内仅在标准化后中性编码，整段日历不删除。系数非零只证明使用了输入，不代表有盈利信息或因果贡献；不按系数挑选下一组特征。

这些是此前已冻结的核心全天路径字段：实现波动、下行占比、尾段风险/收益/成交量、路径效率、首小时后收益、日内回撤/修复、1m/5m风险比例、SOXL相对三倍SOXX的收益残差和方向一致性。未把刚采的未认证资金流或只有较晚覆盖的成分股分钟数据回填进2022年前训练。旧大模型分钟路线仍不晋升；本轮是固定小架构上的条件增量诊断，不是新的独立样本外。

10万美元、冻结IBKR Pro Tiered投影、0/2/5bp独立状态，944区间（2023-01-03信号起至2026-10-09结算）。每个账户单独结算持仓、风控与月度费用，不只从相同收益上扣一个固定成本。

|模型|滑点|累计净收益|收盘最大回撤|盘中监控最大回撤|相对SOXX财富变化|
|---|---|---|---|---|---|
|12项分钟数值+标记|0bp|+303.35%|-28.42%|-29.94%|-17.30%|
|仅可用性标记|0bp|+127.21%|-29.28%|-29.59%|-53.42%|
|不含分钟输入的小模型|0bp|+342.52%|-24.69%|-27.59%|-9.27%|
|原结构匹配参考|0bp|+280.24%|-25.93%|-26.20%|-22.04%|
|原模拟盘模型重建|0bp|+228.99%|-28.11%|-29.50%|-32.55%|
|SOXX|0bp|+387.75%|-42.87%|-43.38%|+0.00%|
|12项分钟数值+标记|2bp|+298.56%|-28.45%|-29.98%|-18.28%|
|仅可用性标记|2bp|+135.83%|-29.44%|-29.76%|-51.65%|
|不含分钟输入的小模型|2bp|+335.36%|-24.74%|-27.64%|-10.74%|
|原结构匹配参考|2bp|+269.16%|-24.40%|-25.77%|-24.31%|
|原模拟盘模型重建|2bp|+279.67%|-24.17%|-27.07%|-22.16%|
|SOXX|2bp|+387.73%|-42.87%|-43.39%|+0.00%|
|12项分钟数值+标记|5bp|+98.09%|-29.45%|-30.07%|-59.38%|
|仅可用性标记|5bp|+326.31%|-29.32%|-29.64%|-12.59%|
|不含分钟输入的小模型|5bp|+348.06%|-24.83%|-27.73%|-8.13%|
|原结构匹配参考|5bp|+266.08%|-24.46%|-25.83%|-24.94%|
|原模拟盘模型重建|5bp|+41.12%|-31.38%|-31.42%|-71.06%|
|SOXX|5bp|+387.70%|-42.88%|-43.39%|+0.00%|

2bp候选+298.56%虽高于仅可用性对照+135.83%，但低于不含分钟输入的强参考+335.36%；5bp候选+98.09%，同时落后可用性对照+326.31%及强参考+348.06%。若只选2bp和较弱对照，会夸大分钟信息作用。5bp盘中监控回撤30.0736%，不能四舍五入成通过30%约束。

|信号年度|候选2bp净收益|相对SOXX财富变化|候选5bp净收益|相对SOXX财富变化|
|---|---|---|---|---|
|2023|+32.01%|-15.71%|+33.72%|-14.60%|
|2024|+21.60%|+0.51%|+21.28%|+0.25%|
|2025|+58.39%|+7.90%|-9.70%|-38.49%|
|2026|+56.77%|-10.61%|+35.27%|-22.87%|

相对财富是增长倍数之比减1，不是收益百分点。不同费用情景对持仓和风控路径的影响显著，不能将更高成本下某个对照收益较高解读为成本有益，也不能只报告最有利情景。原模拟盘模型对照为统一历史重建，并非实际成交收益。

决定：not_promoted_small_head_minute_increment_failed。净SOXX超额、可用性配对全成本、强小模型参考、30%回撤、年度/滚动稳定性和区块置信均失败；独立开发和前向资格也未取得。关闭这套固定小头+核心分钟组，不进一步筛单项、换窗口、删标记、挑seed或调控制阈值挽救。它不证明所有分钟信息都无效，但当前没有理由接入现役模型。

保留不含分钟输入的小模型作为较强的已观察历史参考，仍不晋升。下一项研究需不同的信息机制或明确执行问题证据，不能继续在这组字段上组合搜索。继续历史训练回测，不恢复实时等待；稳定净超额目标未完成。

验证：新增输入零系数、原功能和随机输入前向一致性合同1 passed；真实训练核对三seed初始输出差0、原scaler和成熟标签域一致、分钟字段激活及相对系数非零、完整账户日历一致。NAS冷读回见readback_verification.json。

[完整净费比较](backtest_benchmark_comparison.csv) · [特征使用证据](feature_usage.csv) · [年度](annual_metrics.csv) · [区块区间](uncertainty.csv)

Recorder `b3162bade73b44e7996c9802ecbe7fa7`。
